# 05 - Feature Engineering

**Input:** `data/processed/train_clean.csv` | `val_clean.csv` | `test_clean.csv` (DVC-tracked)

**Output:** `data/processed/train_feat.csv` | `val_feat.csv` | `test_feat.csv` (DVC-tracked)

**Config:** `configs/data_config.yaml` -> `eda_derived.engineered_features`

---

### What this notebook does

| Step | Action | Source |
|---|---|---|
| 1 | Ratio features | EDA multicollinearity analysis (VIF < 10) |
| 2 | Distance features | EDA geographic analysis (SF + LA hubs) |
| 3 | Drop raw size columns | replaced by ratios (inter-corr > 0.9) |
| 4 | Save to `data/processed/` | DVC-tracked |

### No data leakage risk

All transforms are **pure math** (division, sqrt) - no statistics are fit on train.
The same operations are applied independently to each split.

---
## 0 - Setup & Load

In [1]:
import os
import sys
from pathlib import Path

repo_path = Path("/content/california_housing_full_project")
os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

(repo_path / "src" / "__init__.py").touch(exist_ok=True)
(repo_path / "src" / "data" / "__init__.py").touch(exist_ok=True)

import importlib
importlib.invalidate_caches()

print(f"✅ Working dir : {os.getcwd()}")
print(f"✅ sys.path[0] : {sys.path[0]}")

✅ Working dir : /content/california_housing_full_project
✅ sys.path[0] : /content/california_housing_full_project


---
## 1 - Imports

In [3]:
import logging
import pandas as pd
import numpy as np

from src.utils.logger import setup_logging, get_logger
from src.data.data_loader import DataLoader
from src.features.engineering import (
    run_feature_engineering,
    load_feature_config,
    EngineeringResult,
)

setup_logging(level=logging.INFO)
logger = get_logger("notebook.05_feature_engineering")

CONFIG_PATH = "configs/data_config.yaml"

print("Imports ready")

Imports ready


---
## 2 - Pull Cleaned Splits from DVC

The cleaned splits (`train_clean.csv`, `val_clean.csv`, `test_clean.csv`) were
created by `04_cleaning.ipynb` and tracked with DVC.

In [ ]:
import subprocess
from pathlib import Path
from src.utils.paths import IN_COLAB

if IN_COLAB:
    result = subprocess.run(
        ["dvc", "pull", "--remote=mylocal", "data/processed"],
        capture_output=True, text=True, cwd=os.getcwd()
    )
    if result.returncode == 0:
        print("DVC pull successful")
    else:
        print(f"DVC pull warning: {result.stderr.strip()}")
        print("If this is the first run after cleaning, files may already be present.")
else:
    print("Local environment - skipping DVC pull")

# Verify cleaned files exist
for f in ["data/processed/train_clean.csv",
        "data/processed/val_clean.csv",
        "data/processed/test_clean.csv"]:
    p = Path(f)
    status  = "OK" if p.exists() else "MISSING"
    size_mb = p.stat().st_size / 1024**2 if p.exists() else 0
    print(f"  {status}  {f}  ({size_mb:.2f} MB)")

DVC pull successful
  OK  data/processed/train_clean.csv  (1.64 MB)
  OK  data/processed/val_clean.csv  (0.35 MB)
  OK  data/processed/test_clean.csv  (0.35 MB)


---
## 3 - Load Cleaned Splits

> We load all three splits because `run_feature_engineering()` applies the
> same pure-math transforms to each one independently.
> There is no fit step here - no leakage risk.

In [6]:
loader = DataLoader()

train = loader.load_processed("train_clean.csv")
val   = loader.load_processed("val_clean.csv")
test  = loader.load_processed("test_clean.csv")

print(f"train : {train.shape[0]:,} rows x {train.shape[1]} cols")
print(f"val   : {val.shape[0]:,} rows x {val.shape[1]} cols")
print(f"test  : {test.shape[0]:,} rows x {test.shape[1]} cols")
print()
print("Train columns:")
print(train.columns.tolist())

2026-06-22 00:13:55 | INFO     | src.data.data_loader | DataLoader initialized | Drive mode: True
2026-06-22 00:13:55 | INFO     | src.data.data_loader | Loading: /content/california_housing_full_project/data/processed/train_clean.csv
2026-06-22 00:13:55 | INFO     | src.data.data_loader | Loaded 'train_clean.csv' | shape=(14448, 12) | stage=processed
2026-06-22 00:13:55 | INFO     | src.data.data_loader | Loading: /content/california_housing_full_project/data/processed/val_clean.csv
2026-06-22 00:13:55 | INFO     | src.data.data_loader | Loaded 'val_clean.csv' | shape=(3096, 12) | stage=processed
2026-06-22 00:13:55 | INFO     | src.data.data_loader | Loading: /content/california_housing_full_project/data/processed/test_clean.csv
2026-06-22 00:13:55 | INFO     | src.data.data_loader | Loaded 'test_clean.csv' | shape=(3096, 12) | stage=processed
train : 14,448 rows x 12 cols
val   : 3,096 rows x 12 cols
test  : 3,096 rows x 12 cols

Train columns:
['longitude', 'latitude', 'housing_med

---
## 4 - Verify Feature Config

Confirm that `data_config.yaml` has the `eda_derived.engineered_features`
section. If `source = fallback`, the module will use hardcoded defaults
(which mirror the EDA results, but the YAML is the source of truth).

In [7]:
feat_cfg = load_feature_config(CONFIG_PATH)

print(f"Config source  : {feat_cfg.source}")
print()
print("Ratios to create:")
for name, formula in feat_cfg.ratios.items():
    print(f"  {name:<30} = {formula}")
print()
print("Distances to create:")
for name, hub in feat_cfg.distances.items():
    print(f"  {name:<15} -> lat={hub['lat']}, lon={hub['lon']}")
print()
print("Columns to drop after engineering:")
for col in feat_cfg.drop_cols:
    print(f"  {col}")
print()
if feat_cfg.source == "fallback":
    print("WARNING: eda_derived.engineered_features missing from data_config.yaml")
    print("Using hardcoded fallback defaults (values match last EDA run).")
else:
    print("Config loaded from data_config.yaml - ready to engineer")

2026-06-22 00:14:05 | INFO     | src.features.engineering | Feature config loaded: 3 ratios, 2 distances, 4 cols to drop
Config source  : config

Ratios to create:
  rooms_per_household            = total_rooms / households
  bedrooms_per_room              = total_bedrooms / total_rooms
  population_per_household       = population / households

Distances to create:
  dist_SF         -> lat=37.77, lon=-122.42
  dist_LA         -> lat=34.05, lon=-118.24

Columns to drop after engineering:
  total_rooms
  total_bedrooms
  population
  households

Config loaded from data_config.yaml - ready to engineer


---
## 5 - Run Feature Engineering

`run_feature_engineering()` applies three steps in order:
1. **Ratio features** - divide count columns to reduce multicollinearity
2. **Distance features** - Euclidean distance to SF and LA price hubs
3. **Drop raw columns** - remove the raw size columns replaced by ratios

In [8]:
result = run_feature_engineering(
    train=train,
    val=val,
    test=test,
    config_path=CONFIG_PATH,
    auto_track_dvc=True,
)

print(result.summary())

2026-06-22 00:14:22 | INFO     | src.features.engineering | ============================================================
2026-06-22 00:14:22 | INFO     | src.features.engineering |   Feature engineering started
2026-06-22 00:14:22 | INFO     | src.features.engineering | ============================================================
2026-06-22 00:14:22 | INFO     | src.features.engineering | Feature config loaded: 3 ratios, 2 distances, 4 cols to drop
2026-06-22 00:14:22 | INFO     | src.features.engineering | Step 1/3 — Ratio features
2026-06-22 00:14:22 | INFO     | src.features.engineering | Created ratio 'rooms_per_household' = total_rooms / households | mean=1.282 | nulls=0
2026-06-22 00:14:22 | INFO     | src.features.engineering | Created ratio 'bedrooms_per_room' = total_bedrooms / total_rooms | mean=0.792 | nulls=0
2026-06-22 00:14:22 | INFO     | src.features.engineering | Created ratio 'population_per_household' = population / households | mean=1.179 | nulls=0
2026-06-22 00:14:

---
## 6 - Verify Results

Four checks:
- **New features present** in all splits
- **Raw columns dropped** from all splits
- **No nulls** in new features
- **No inf values** in ratio features

In [9]:
print("-- New features check --")
new_cols = list(feat_cfg.ratios.keys()) + list(feat_cfg.distances.keys())

for col in new_cols:
    for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
        status = "OK" if col in df.columns else "MISSING"
        print(f"  {col:<30} in {name:<6} : {status}")

-- New features check --
  rooms_per_household            in train  : OK
  rooms_per_household            in val    : OK
  rooms_per_household            in test   : OK
  bedrooms_per_room              in train  : OK
  bedrooms_per_room              in val    : OK
  bedrooms_per_room              in test   : OK
  population_per_household       in train  : OK
  population_per_household       in val    : OK
  population_per_household       in test   : OK
  dist_SF                        in train  : OK
  dist_SF                        in val    : OK
  dist_SF                        in test   : OK
  dist_LA                        in train  : OK
  dist_LA                        in val    : OK
  dist_LA                        in test   : OK


In [10]:
print("-- Dropped columns check --")
for col in feat_cfg.drop_cols:
    for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
        still_present = col in df.columns
        status = "STILL PRESENT (unexpected)" if still_present else "OK (dropped)"
        print(f"  {col:<20} in {name:<6} : {status}")

-- Dropped columns check --
  total_rooms          in train  : OK (dropped)
  total_rooms          in val    : OK (dropped)
  total_rooms          in test   : OK (dropped)
  total_bedrooms       in train  : OK (dropped)
  total_bedrooms       in val    : OK (dropped)
  total_bedrooms       in test   : OK (dropped)
  population           in train  : OK (dropped)
  population           in val    : OK (dropped)
  population           in test   : OK (dropped)
  households           in train  : OK (dropped)
  households           in val    : OK (dropped)
  households           in test   : OK (dropped)


In [11]:
print("-- Null check in new features --")
for col in new_cols:
    for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
        if col not in df.columns:
            continue
        nulls = df[col].isnull().sum()
        status = "OK" if nulls == 0 else f"WARN ({nulls} nulls)"
        print(f"  {col:<30} {name:<6} : {status}")

-- Null check in new features --
  rooms_per_household            train  : OK
  rooms_per_household            val    : OK
  rooms_per_household            test   : OK
  bedrooms_per_room              train  : OK
  bedrooms_per_room              val    : OK
  bedrooms_per_room              test   : OK
  population_per_household       train  : OK
  population_per_household       val    : OK
  population_per_household       test   : OK
  dist_SF                        train  : OK
  dist_SF                        val    : OK
  dist_SF                        test   : OK
  dist_LA                        train  : OK
  dist_LA                        val    : OK
  dist_LA                        test   : OK


In [12]:
import numpy as np

print("-- Inf check in ratio features --")
for col in feat_cfg.ratios.keys():
    for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
        if col not in df.columns:
            continue
        n_inf = np.isinf(df[col]).sum()
        status = "OK" if n_inf == 0 else f"WARN ({n_inf} inf values)"
        print(f"  {col:<30} {name:<6} : {status}")

-- Inf check in ratio features --
  rooms_per_household            train  : OK
  rooms_per_household            val    : OK
  rooms_per_household            test   : OK
  bedrooms_per_room              train  : OK
  bedrooms_per_room              val    : OK
  bedrooms_per_room              test   : OK
  population_per_household       train  : OK
  population_per_household       val    : OK
  population_per_household       test   : OK


---
## 7 - Feature Statistics

Quick stats on the new features - train only.

In [13]:
print("-- New feature statistics (train) --")
stats = result.train[new_cols].describe().T[
    ["mean", "std", "min", "max"]
].round(4)
print(stats.to_string())

-- New feature statistics (train) --
                            mean     std     min     max
rooms_per_household       1.2818  0.0800  0.9416  3.5609
bedrooms_per_room         0.7918  0.0377  0.3155  1.2037
population_per_household  1.1794  0.0762  0.8614  4.5827
dist_SF                   3.8555  2.5041  0.0000  9.3079
dist_LA                   2.6726  2.4185  0.0000  9.8600


---
## 8 - Schema Consistency

In [14]:
print("-- Shape check --")
for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
    print(f"  {name:<6} : {df.shape[0]:,} rows x {df.shape[1]} cols")

print()
print("-- Column consistency --")
train_cols = list(result.train.columns)
for name, df in [("val", result.val), ("test", result.test)]:
    if list(df.columns) == train_cols:
        print(f"  {name} columns match train - OK")
    else:
        diff = set(train_cols) ^ set(df.columns)
        print(f"  {name} column mismatch: {diff}")

print()
print("-- Final column list --")
print(result.train.columns.tolist())

-- Shape check --
  train  : 14,448 rows x 13 cols
  val    : 3,096 rows x 13 cols
  test   : 3,096 rows x 13 cols

-- Column consistency --
  val columns match train - OK
  test columns match train - OK

-- Final column list --
['longitude', 'latitude', 'housing_median_age', 'median_income', 'median_house_value', 'ocean_proximity', 'is_capped', 'lof_outlier', 'rooms_per_household', 'bedrooms_per_room', 'population_per_household', 'dist_SF', 'dist_LA']


---
## 9 - DVC Tracking Confirmation

In [15]:
from pathlib import Path

print("-- DVC pointer files --")
for f in ["data/processed.dvc", "data/.gitignore"]:
    status = "OK" if Path(f).exists() else "MISSING"
    print(f"  {status}  {f}")

print()
print("-- Saved files in data/processed/ --")
for f in sorted(Path("data/processed").glob("*.csv")):
    size_mb = f.stat().st_size / 1024**2
    tag = "feat" if "feat" in f.name else "clean"
    print(f"  [{tag}]  {f.name:<25}  {size_mb:.2f} MB")

print()
print(f"DVC tracked: {'yes' if result.dvc_tracked else 'no - run dvc push manually'}")

if result.warnings:
    print()
    print("Warnings:")
    for w in result.warnings:
        print(f"  ! {w}")

-- DVC pointer files --
  OK  data/processed.dvc
  OK  data/.gitignore

-- Saved files in data/processed/ --
  [clean]  test_clean.csv             0.35 MB
  [feat]  test_feat.csv              0.42 MB
  [clean]  train_clean.csv            1.64 MB
  [feat]  train_feat.csv             1.94 MB
  [clean]  val_clean.csv              0.35 MB
  [feat]  val_feat.csv               0.42 MB

DVC tracked: yes


---
## 10 - Git Commit

In [16]:
print("Run in terminal:")
print()
print("  git add data/processed.dvc data/.gitignore")
print('  git commit -m "data: add feature-engineered splits (DVC-tracked)"')
print("  git push")

Run in terminal:

  git add data/processed.dvc data/.gitignore
  git commit -m "data: add feature-engineered splits (DVC-tracked)"
  git push


---
## Summary & Next Steps

| Done | Details |
|---|---|
| Ratio features | `rooms_per_household`, `bedrooms_per_room`, `population_per_household` |
| Distance features | `dist_SF`, `dist_LA` |
| Raw cols dropped | `total_rooms`, `total_bedrooms`, `population`, `households` |
| Files saved | `data/processed/*_feat.csv` (DVC-tracked) |
